In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS quickcart.quarantine;

###Import required libraries

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

#####Data Profiling

In [0]:
# customers_bronze = spark.table("""
#                             quickcart.bronze.customers
#                             """)

# #customer_bronze.printSchema()

# print("Bronze Customers:", customers_bronze.count())

#Checking NULLS

# customers_bronze.select([
#     F.count(F.when(F.col(c).isNull(), c)).alias(c) 
#     for c in customers_bronze.columns]).display()

#Duplicate records

# customers_bronze.groupBy(col("customer_id")).agg(count(col("customer_id")).alias("c"))\
#     .filter(col("c")>1).display()

#Check invalid customer IDs

#customers_bronze.filter(col("customer_id").isNull()).count()
#customers_bronze.filter(trim(col("customer_id")) == "").count()

#Check email quality

# customers_bronze.filter(
#     ~F.col("email").rlike(
#         r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
#     )
# ).select(
#     "customer_id",
#     "email"
# ).display()



###Create Silver configuration

In [0]:
SILVER_CONFIG = {
    "customers" : {
        "bronze_table" : "quickcart.bronze.customers",
        "silver_table" : "quickcart.silver.customers",
        "quarantine_table" : "quickcart.quarantine.customers",
        "primary_key" : "customer_id",
        "watermark_column" : "updated_at",
    },
    "products" : {
        "bronze_table" : "quickcart.bronze.products",
        "silver_table" : "quickcart.silver.products",
        "quarantine_table" : "quickcart.quarantine.products",
        "primary_key" : "product_id",
        "watermark_column" : "updated_at",
    },
    "orders" : {
        "bronze_table" : "quickcart.bronze.orders",
        "silver_table" : "quickcart.silver.orders",
        "quarantine_table" : "quickcart.quarantine.orders",
        "primary_key" : "order_id",
        "watermark_column" : "updated_at",
    },
    "payments" : {
        "bronze_table" : "quickcart.bronze.payments",
        "silver_table" : "quickcart.silver.payments",
        "quarantine_table" : "quickcart.quarantine.payments",
        "primary_key" : "payment_id",
        "watermark_column" : "updated_at",
    },
    "deliveries" : {
        "bronze_table" : "quickcart.bronze.deliveries",
        "silver_table" : "quickcart.silver.deliveries",
        "quarantine_table" : "quickcart.quarantine.deliveries",
        "primary_key" : "delivery_id",
        "watermark_column" : "updated_at",
    }
}

###Test the configuration

In [0]:
config = SILVER_CONFIG['customers']

print(config)

###Add a table validation function

In [0]:
def validate_table_config(table_name):
    if table_name not in SILVER_CONFIG:
        raise ValueError(
            f"Unsupported table: {table_name}. "
            f"Supported tables: {list(SILVER_CONFIG.keys())}"
            ) 
    return SILVER_CONFIG[table_name]

#print(validate_table_config("customers"))

###Create the standardization function

In [0]:
def standardize_columns(table_name, df):
    if table_name == "Customers":
        df = df\
            .withColumn("customer_id", F.trim(F.col("customer_id")))\
                .withColumn("customer_name", F.trim(F.col("customer_name")))\
                    .withColumn("email", F.lower(F.trim(F.col("email"))))\
                        .withColumn("phone", F.regexp_replace(F.trim(F.col("phone")),r"\s+",""))\
                            .withColumn("gender", F.upper(F.trim(F.col("gender"))))\
                                .withColumn("city", F.initcap(F.trim(F.col("city"))))\
                                    .withColumn("state", F.initcap(F.trim(F.col("state"))))\
                                        .withColumn("pincode", F.trim(F.col("pincode")))\
                                            .withColumn("customer_segment", F.upper(F.trim(F.col("customer_name"))))
    elif table_name == "Products":
        pass
    elif table_name == "Orders":
        pass
    elif table_name == "Payments":
        pass
    elif table_name == "Deliveries":
        pass
    return df

###Create the framework skeleton

In [0]:
def process_to_silver(table_name):
    print("=" * 70)
    print(f"Starting Silver processing: {table_name}")
    print("=" * 70)

    config = validate_table_config(table_name)
    bronze_table = config["bronze_table"]
    silver_table = config["silver_table"]
    quarantine_table = config["quarantine_table"]
    primary_key = config["primary_key"]
    watermark_column = config["watermark_column"]

    print(f"Bronze Table      : {bronze_table}")
    print(f"Silver Table      : {silver_table}")
    print(f"Quarantine Table  : {quarantine_table}")
    print(f"Primary Key       : {primary_key}")
    print(f"Watermark Column  : {watermark_column}")

    bronze_df = spark.table(bronze_table)

    print(
        f"Bronze Record Count: {bronze_df.count()}"
    )

    standardized_df = standardize_columns(
        table_name,
        bronze_df
    )

    print(
        "Standardization completed successfully."
    )

    # ----------------------------------------
    # Transformation logic will be added here
    # ----------------------------------------
    return standardized_df


In [0]:
customers_standardized = process_to_silver("orders")
display(
    customers_standardized.limit(10)
)